# Overnight queue: train several cohorts back to back (resumable)

Runs the configs in `QUEUE` one after another within one time budget (`TOTAL_HOURS`). Each cohort restores its
finished adapters from **its own** private HF repo, trains the rest, uploads every 15 minutes (only its own
adapters, to its own repo), prints an ASR check and exports its manifest. If time runs out, the next launch resumes.

**How to run:** GPU = L4. Run cells 1-3 one by one (cell 2: type `z` and upload `code_phase2.zip`; cell 3: HF token),
then *Runtime -> Run cell and below* from cell 4. Keep the laptop awake and the tab open; the runtime disconnects
itself at the end if every upload worked.

Rough lengths on an L4: calib_v1 2.5 h, train_v1 8-9 h, eval_v1 2.5 h, final_v1 2 h.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
# 2. Code + packages. The team repo is private, so either
#    (a) upload code_phase2.zip when asked (made on the laptop with `git archive`; no GitHub token needed), or
#    (b) paste a GitHub token (classic, 'repo' scope; starts with ghp_). NOT the Hugging Face token (hf_...).
import os, subprocess, zipfile, shutil
from getpass import getpass
BRANCH = 'natour/adapters-payloads'
REPO_URL = 'https://github.com/JuliaIssa2/FYP-AI-Antivirus.git'
DEST = '/content/FYP-AI-Antivirus'
NOPROMPT = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}

def clone(url):  # never let the URL (which may hold a token) reach an error message
    return subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DEST], capture_output=True, text=True, env=NOPROMPT).returncode == 0

if os.path.exists(DEST + '/.git'):
    subprocess.run(['git', '-C', DEST, 'pull', '-q'], env=NOPROMPT, check=False)
    COMMIT = subprocess.check_output(['git', '-C', DEST, 'rev-parse', 'HEAD'], text=True).strip()
elif os.path.exists(DEST + '/adapter_generation') and os.path.exists(DEST + '/CODE_VERSION'):
    COMMIT = open(DEST + '/CODE_VERSION').read().strip()          # unpacked from a zip earlier in this runtime
elif clone(REPO_URL):
    COMMIT = subprocess.check_output(['git', '-C', DEST, 'rev-parse', 'HEAD'], text=True).strip()
else:
    choice = input('Repo is private. Type z to upload code_phase2.zip, or g to paste a GitHub token: ').strip().lower()
    if choice == 'g':
        gh = getpass('GitHub token (ghp_... or github_pat_...): ').strip()
        if gh.startswith('hf_'):
            raise SystemExit('That is a Hugging Face token. Use a GitHub token, or choose z.')
        ok = clone(REPO_URL.replace('https://', f'https://x-access-token:{gh}@')); del gh
        if not ok:
            raise SystemExit('GitHub refused the token (needs read access to JuliaIssa2/FYP-AI-Antivirus). Choose z instead.')
        subprocess.run(['git', '-C', DEST, 'remote', 'set-url', 'origin', REPO_URL], check=True)  # do not keep the token on disk
        COMMIT = subprocess.check_output(['git', '-C', DEST, 'rev-parse', 'HEAD'], text=True).strip()
    else:
        from google.colab import files
        up = files.upload(); name = list(up)[0]
        shutil.rmtree(DEST, ignore_errors=True)
        with zipfile.ZipFile(name) as z:
            z.extractall(DEST)
            COMMIT = z.comment.decode().strip() or 'zip-unknown'   # git archive stores the commit id here
        open(DEST + '/CODE_VERSION', 'w').write(COMMIT)
os.chdir(DEST + '/adapter_generation')
print('code commit:', COMMIT)
!pip -q install transformers peft datasets accelerate pyyaml safetensors pandas huggingface_hub
!pip -q uninstall -y torchao

In [ ]:
# 3. Settings
HF_WRITE_TOKEN = getpass('HF WRITE token: ').strip()
QUEUE = [                                   # (config, run id): run ids must be new for every launch
    ('configs/train_v1.yaml', 'train-a'),
    ('configs/eval_v1.yaml',  'eval-a'),
    ('configs/final_v1.yaml', 'final-a'),   # uploads to its own repo (hf_repo: adapterlab-final)
]
TOTAL_HOURS = 11.0                          # no new adapter starts after this; unfinished cohorts resume next launch
SHUTDOWN_WHEN_DONE = True

In [ ]:
# 4. Run the queue
import sys, time
sys.path.insert(0, '.')
import colab_runner
deadline = time.time() + TOTAL_HOURS * 3600
results = {}
for config, run_id in QUEUE:
    status = colab_runner.run_cohort(config, run_id, HF_WRITE_TOKEN, COMMIT, deadline)
    results[run_id] = status
    if status in ('failed', 'upload_failed'):
        print('stopping the queue:', run_id, status); break
print('\nQUEUE RESULT:', results)

In [ ]:
# 5. Stop spending compute units
if SHUTDOWN_WHEN_DONE and results and all(s in ('done', 'partial') for s in results.values()):
    print('All uploads done. Disconnecting the runtime.')
    from google.colab import runtime; runtime.unassign()
else:
    print('NOT disconnected (an upload failed, a cohort failed, or SHUTDOWN_WHEN_DONE is False):', results)